In [ ]:
import yaml
import json
from IPython.display import display, HTML

# def read_all_yaml_files(path: str) -> List[Dict[str, Any]]:
#   result = []
#   for file in glob.iglob(path):
#     with open(file, 'r') as fh:
#       result.append(yaml.safe_load(fh))
#   return result

# def merge_yaml_schema(schema: List[Dict[str, Any]]) -> Dict[str, Any]:
#   result = {}
#   for s in schema:
#     print("parsing...")
#     for k, v in s.items():
#       if k not in result:
#         result[k] = v
#       elif isinstance(v, dict):
#         result[k] = merge_yaml_schema([result[k], v])
#       else:
#         # result[k] = type(v)([result[k], v])
#         result[k] = [result[k], v]
#   return result

def merge_values(data1, data2):
  """
  Recursively merge values from two YAML data structures.
  Lists are concatenated, and dictionaries are merged.
  """
  if isinstance(data1, dict) and isinstance(data2, dict):
    for key2, value2 in data2.items():
      if key2 in data1:
        value1 = data1[key2]
        if isinstance(value1, dict) != isinstance(value2, dict):
          if isinstance(value1, dict):
            value2 = {"##no-key": value2}
          else:
            value1 = {"##no-key": value1}
        data1[key2] = merge_values(value1, value2)
      else:
        data1[key2] = value2
    return data1
  elif isinstance(data1, list) and isinstance(data2, list):
    return data1 + data2
  else:
    if data1 is None:
      res = data2
    elif isinstance(data1, list):
      uniq_vals = set(data1 + [data2])
      res = list(sorted(uniq_vals, key=lambda x: (x is None, x)))
    else:
      res = sorted([data1, data2], key=lambda x: (x is None, x))
    return res

# barracks_yaml_files = read_all_yaml_files(r'S:\src\unknown-horizons\content\objects\**\*.yaml')
from pathlib import Path

files = list(Path(r's:\src\unknown-horizons\content\objects').glob(r'**\*.yaml'))
res = {}
for file in files:
  print(f"Merging {file}")
  with open(file, 'r') as fh:
    obj = yaml.safe_load(fh)
    res = merge_values(res, obj)
display(HTML("<pre>"+json.dumps(res, indent=2)+"</pre>"))
print("Done")

In [ ]:
from pathlib import Path, PurePosixPath
from jinja2 import Environment, FileSystemLoader
from PIL import Image, ImageOps
import yaml
import re

def rt(template_str: str, args: dict):
  env = Environment(loader=FileSystemLoader("."), trim_blocks=True) # trim blocks remove extra newline around for loops and other blocks
  template = env.from_string(template_str.strip())
  rendered_content = template.render(args)
  return rendered_content
def write_template(output_path: Path, template_str: str, args: dict):
  print(f"Writing template: {output_path}")
  rendered_content = rt(template_str, args)
  output_path.write_text(rendered_content, newline="\n")
def generate_uid(building_name: str, file_id: str) -> str: # generates valid Godot UID for the given building name and file id
  valid_uid: str = "c"
  # add the building name to the uid
  stripped_building_name: str = building_name.lower().replace("_", "").replace("z", "").replace("9", "") # strip the building name of unallowed characters
  building_name_cutted: str = stripped_building_name[:min(len(stripped_building_name), 10)] # cut the building name to leave room for the first letter of id
  valid_uid += building_name_cutted
  # add the file id to the uid
  stripped_file_id = file_id.replace("_", "").lower().replace("z", "").replace("9", "") # strip the file id of unallowed characters
  file_id_cutted = stripped_file_id[:min(len(stripped_file_id), 13 - len(valid_uid))] # cut the file id to fit the remaining space in the uid to 13 characters
  valid_uid += file_id_cutted
  valid_uid = valid_uid.ljust(13, "0") # add a spacer to make the uid 13 characters
  print(f"Generated uid: {valid_uid}")
  return valid_uid

In [ ]:
bakery_tscn_template = """
[gd_scene load_steps=8 format=3 uid="uid://{{ tscn_uid }}"]

{% if baseclass.startswith("collectors.") %}
[ext_resource type="Script" uid="uid://dryewpxj1qix4" path="res://Assets/World/Components/Collectors/Collector.gd" id="1_script_gd"]{% else %}
[ext_resource type="Script" uid="uid://4liotsbcpcls" path="res://Assets/World/Buildings/Building2D.gd" id="1_script_gd"]
{% endif %}
[ext_resource type="PackedScene" uid="uid://x1upwhg1f71a" path="res://Assets/World/Components/Selectable/Selectable.tscn" id="2_selectable_component"]
{% if components.AmbientSoundComponent is defined and components.AmbientSoundComponent.soundfiles is defined %}
[ext_resource type="PackedScene" uid="uid://kv52rh351xud" path="res://Assets/World/Components/AmbientSoundComponent/AmbientSoundComponent.tscn" id="3_ambient_sound_component"]
{% endif %}
[ext_resource type="PackedScene" uid="uid://c7w3xnajww1kq" path="res://Assets/World/Components/BuildingActionSet/BuildingActionSet.tscn" id="3_building_action_set_component"]
[ext_resource type="SpriteFrames" uid="uid://{{ tres_uid }}" path="res://{{ sprite_frames_tres_path }}" id="4_sprite_frames"]
{% if components.HealthComponent is defined %}
[ext_resource type="PackedScene" uid="uid://cqu1iyo1pym8w" path="res://Assets/World/Components/HealthComponent/HealthComponent.tscn" id="5_health_component"]
{% endif %}
{% if components.StorageComponent is defined and components.StorageComponent.PositiveSizedSlotStorage is defined %}
[ext_resource type="PackedScene" uid="uid://b3hix4pdlnumi" path="res://Assets/World/Components/Storages/SizedStorageComponent/SizedStorageComponent.tscn" id="5_sized_storage_component"]
{% endif %}
{% if components.StorageComponent is defined and components.StorageComponent.SlotsStorage is defined %}
[ext_resource type="PackedScene" uid="uid://ck36mqeae5ana" path="res://Assets/World/Components/Storages/SlotStorageComponent/SlotStorageComponent.tscn" id="5_slot_storage_component"]
{% endif %}
{% if components.StorageComponent is defined and components.StorageComponent.SettlementStorage is defined %}
[ext_resource type="PackedScene" uid="uid://bw7vhkspeuvnf" path="res://Assets/World/Components/Storages/SettlementStorageComponent/SettlementStorageComponent.tscn" id="5_settlement_storage_component"]
{% endif %}
{% if components.ProducerComponent is defined %}
[ext_resource type="PackedScene" uid="uid://bo27kwd5m1jmc" path="res://Assets/World/Components/ProductionLine/ProductionLineComponent.tscn" id="6_production_line_component"]
{% endif %}
{% for collector_name, count in components.get("CollectingComponent", {}).get("collectors", {}).items() %}
{% set collector_strid = collector_name | replace("UNITS.", "") | lower %}
[ext_resource type="PackedScene" uid="uid://{{ get_uuid("u" + collector_strid, "tscn") }}" path="res://Assets/World/Units2/{{ collector_strid }}/{{ collector_strid }}.tscn" id="7_{{ collector_strid }}"]
{% endfor %}
{% if baseclass.startswith("production.") %}
[ext_resource type="PackedScene" uid="uid://cubuildingcts" path="res://Assets/World/Units2/building_collector/building_collector.tscn" id="9_building_collector_component"]
{% endif %}
{% if baseclass.startswith("collectors.") %}
[ext_resource type="PackedScene" uid="uid://unr7kegy5cn4" path="res://Assets/World/Components/MoveByCell/MoveByCellComponent.tscn" id="10_move_by_cell_component"]
{% endif %}

[node name="{{ name | replace("'", "\\\\'") | replace(".", "_") | replace(":", "_") }}" type="Node2D"]
script = ExtResource("1_script_gd")
baseclass = "{{ baseclass }}"
radius = {{ radius }}
{% if cost              is defined -%}  cost = {{ cost }}                                   {{-"\n"}}{% endif %}
{% if cost_inactive     is defined -%}  cost_inactive = {{ cost_inactive }}                 {{-"\n"}}{% endif %}
{% if size_x            is defined -%}  size = Vector2i({{ size_x }}, {{ size_y }})         {{-"\n"}}{% endif %}
{% if inhabitants       is defined -%}  inhabitants = {{ inhabitants }}                     {{-"\n"}}{% endif %}
{% if tooltip_text      is defined -%}  tooltip_text = "{{ tooltip_text }}"                 {{-"\n"}}{% endif %}
{# buildingcosts are in BuildingConfig #}
{% if show_status_icons is defined %}   show_status_icons = {{ show_status_icons | lower }} {{-"\n"}}{% endif %}
{% if tier              is defined -%}  tier = "{{ tier }}"                                 {{-"\n"}}{% endif %}
{% if velocity          is defined -%}  velocity = {{ velocity }}                           {{-"\n"}}{% endif %}

{% if components.SelectableComponent is defined %}
[node name="Selectable" parent="." instance=ExtResource("2_selectable_component")]
type = "{{ components.SelectableComponent.type }}"
tabs = Array[String]({{ components.SelectableComponent.tabs | tojson }})
enemy_tabs = Array[String]({{ components.SelectableComponent.enemy_tabs | tojson  }})

{% endif %}
{# --------------------------------------------------------------------------------------- #}
{% if components.HealthComponent is defined %}
[node name="HealthComponent" parent="." instance=ExtResource("5_health_component")]
max_health = {{ components.HealthComponent.maxhealth }}

{% endif %}
{# --------------------------------------------------------------------------------------- #}
{% if components.ProducerComponent is defined %}
{% for line_name, line in components.ProducerComponent.productionlines.items() %}
[node name="ProductionLineComponent_{{ line_name }}" parent="." instance=ExtResource("6_production_line_component")]
line_name = "{{line_name}}"
consumes = Dictionary[StringName, int]({{ line.consumes | tojson | replace('"RES.', '\n&"') | replace(': -', ': ') | replace('}', '\n}') }})
produces = Dictionary[StringName, int]({{ line.produces | tojson | replace('"RES.', '\n&"') | replace('}', '\n}') }})

{% endfor %}
{% endif %}
{# --------------------------------------------------------------------------------------- #}
{% if components.StorageComponent is defined %}
{% if components.StorageComponent.SlotsStorage is defined %}
[node name="SlotStorageComponent" parent="." instance=ExtResource("5_slot_storage_component")]
max_capacity = Dictionary[StringName, int]({{ components.StorageComponent.SlotsStorage.slot_sizes | tojson | replace('"RES.', '\n&"') | replace('}', '\n}') }})

{% endif %}
{% if components.StorageComponent.SettlementStorage is defined %}
[node name="SettlementStorageComponent" parent="." instance=ExtResource("5_settlement_storage_component")]

{% endif %}
{% if components.StorageComponent.PositiveSizedSlotStorage is defined %}
[node name="SizedStorageComponent" parent="." instance=ExtResource("5_sized_storage_component")]
limit = {{ components.StorageComponent.PositiveSizedSlotStorage.limit }}

{% endif %}
{% endif %}
{# --------------------------------------------------------------------------------------- #}
{% for collector_name, count in components.get("CollectingComponent", {}).get("collectors", {}).items() %}
{% for i in range(count) %}
{% set collector_strid = collector_name | replace("UNITS.", "") | lower %}
[node name="{{ collector_strid }}_{{ i }}" parent="." instance=ExtResource("7_{{ collector_strid }}")]

{% endfor %}
{% endfor %}
{# --------------------------------------------------------------------------------------- #}
{% if baseclass.startswith("production.") %}
[node name="BuildingCollector" parent="." instance=ExtResource("9_building_collector_component")]
{% endif %}
{# --------------------------------------------------------------------------------------- #}
{% if baseclass.startswith("collectors.") %}
[node name="MoveByCellComponent" parent="." instance=ExtResource("10_move_by_cell_component")]

{% endif %}
{# --------------------------------------------------------------------------------------- #}
{% if components.AmbientSoundComponent is defined and components.AmbientSoundComponent.soundfiles is defined %}
[node name="AmbientSoundComponent" parent="." instance=ExtResource("3_ambient_sound_component")]
sound_files = Array[String]({{ components.AmbientSoundComponent.soundfiles | tojson }})

{% endif %}
{# --------------------------------------------------------------------------------------- #}
[node name="BuildingActionSet" parent="." instance=ExtResource("3_building_action_set_component")]
sprite_frames = ExtResource("4_sprite_frames")

"""

animation_frames_tres_template = """
[gd_resource type="SpriteFrames" load_steps=39 format=3 uid="uid://{{ uid }}"]

{% set unique_source_atlas_paths = sprite_frames_animations.values() | map(attribute='source_atlas_path') | list | unique %}
{% for fa in unique_source_atlas_paths | sort %}
[ext_resource type="Texture2D" uid="uid://{{ fp2uid[fa] }}" path="res://{{ fa }}" id="{{ fa.stem }}"]
{% endfor %}

{% for animation_name, sfa in sprite_frames_animations.items() | sort %}
{% for frame_pos in sfa["animation_frames_positions"] %}
[sub_resource type="AtlasTexture" id="{{ animation_name | replace(".", "_") }}_{{  frame_pos[0] }}_{{ frame_pos[1] }}"]
atlas = ExtResource("{{ sfa["source_atlas_path"].stem }}")
region = Rect2({{ frame_pos[0] }}, {{ frame_pos[1] }}, {{ frame_pos[2] }}, {{ frame_pos[3] }})

{% endfor %}
{% endfor -%}

[resource]
animations = [
{%- for animation_name, sfa in sprite_frames_animations.items() | sort -%}
{
"frames": [
{%- for frame_pos in sfa["animation_frames_positions"] -%}
{
"duration": 1.0,
"texture": SubResource("{{ animation_name | replace(".", "_") }}_{{ frame_pos[0] }}_{{ frame_pos[1] }}")
}{% if not loop.last %},{% endif -%}
{%- endfor -%}
],
"loop": true,
"name": &"{{ animation_name }}",
"speed": 5.0
}{% if not loop.last %}, {% endif -%}
{%- endfor %}
]{{ "\n" }}
"""

def process_action_sets(action_sets: dict[str, dict[str, None]], object_name: str, save_path: Path):
  animations = {}
  for tier, as_names in action_sets.items():
    for as_name, _ in as_names.items():
      # print(tier, as_name)
      # action set folder is impossible to figure out based on the yaml file. Search for the corresponding folder:
      as_folder_candidates = list(gfx_path.parent.glob(f"**/{as_name}"))
      assert len(as_folder_candidates) == 1 and as_folder_candidates[0].exists()
      as_folder = as_folder_candidates[0]
      # print(as_folder, as_folder.exists())
      # as_folder = Path(r"S:\src\unknown-horizons\content\gfx\buildings\pioneers\brewery\as_brewery0") # !!!
      diff = list(set(as_folder.glob("**/*.png")) - set(as_folder.glob("**/*.*")))
      if diff:
        raise Exception(f"UNEXPECTED diff: {diff}")
      for work_idle_path in as_folder.iterdir():
        if work_idle_path.stem == "deleteme": continue # some extra file
        sprite_data = {} # one sprite per type (WORK or IDLE or IDLE_FULL). Each sprite contains 4 rows of animation frames
        for angle_path in work_idle_path.iterdir():
          if angle_path.stem.startswith("tm_") and angle_path.is_file(): continue # some extra file
          angle = angle_path.stem.rjust(3, "0") # '45' => '045'
          sprite_data[angle] = sprite_datum = {
            "row_width": 0,
            "row_height": 0,
            "animation_frames": []
          }
          # sprite_datum["animation_frames"] = []
          for img_path in angle_path.iterdir():
            img = Image.open(img_path)
            sprite_datum["row_width"] += img.width
            sprite_datum["row_height"] = max(sprite_datum["row_height"], img.height)
            sprite_datum["animation_frames"].append({"path": img_path, "img": img})
            print(work_idle_path.stem, angle, img_path)
        total_width = max(sd["row_width"] for sd in sprite_data.values())
        total_height = sum(sd["row_height"] for sd in sprite_data.values())
        sprite_img = Image.new("RGBA", (total_width, total_height)) # each animation frame set on a separate row
        y = 0
        for angle, sprite_datum in sorted(sprite_data.items()):
          x = 0
          for frame in sprite_datum["animation_frames"]:
            sprite_img.paste(frame["img"], (x, y))
            frame["xywh"] = (x, y, *frame["img"].size)
            x += frame["img"].width
          y += sprite_datum["row_height"]
        # str(work_idle_path.relative_to(work_idle_path.parent.parent.parent)).replace("\\","_")
        file_name = (object_name + "_" + # eg. 'bakery_'
                    tier.removeprefix("TIER.").lower() + "_" +            # eg. 'sailors_'
                    as_name + "_" +                                       # eg. 'as_brewery0_'
                    work_idle_path.stem)                                  # eg. 'idle'
        file_path = save_path / f"{file_name}.png"
        print("Saving to ", file_path)
        sprite_img.save(file_path)
        # display(sprite_img)

        # get uid if exists:
        import_file_path = file_path.with_name(file_path.name + ".import")
        file_path_uid = None
        if import_file_path.exists():
          import_file_text = import_file_path.read_text()
          if (m := re.search(r'uid="uid://(?P<uid>[^"]+)"', import_file_text)) != None:
            file_path_uid = m.group("uid")
        
        # save data for a single tres for all sprite frames for a particular building
        for angle, sprite_datum in sprite_data.items():
          animation_name = as_name.removeprefix("as_") + "." + tier.removeprefix("TIER.").lower() + "." + work_idle_path.stem + "." + angle
          if animation_name in animations:
            raise Exception(f"DUPLICATE ANIMATION NAME: {animation_name}")
          animations[animation_name] = {
            "source_atlas_path": PurePosixPath(file_path.relative_to(project_path)),
            "animation_frames_positions": [sd["xywh"] for sd in sprite_datum["animation_frames"]],
            "source_atlas_path_uid": file_path_uid
          }
        print(as_folder / f"{work_idle_path.stem}.png")

  return animations

project_path = Path("s:/src/unknown-horizons-godot-port")
objects_save_path = project_path / "Assets/World" # Units2 and Buildings2 folders

uh_original_objects_path = Path(r"S:\src\unknown-horizons\content\objects")
# yaml_files = list(uh_original_objects_path.glob("buildings/warehouse.yaml"))
# yaml_files = list(uh_original_objects_path.glob("buildings/pastryshop.yaml")
# yaml_files = list(uh_original_objects_path.glob("buildings/tent.yaml"))
# yaml_files = list(uh_original_objects_path.glob("buildings/lumberjackcamp.yaml"))
# yaml_files = list(uh_original_objects_path.glob("buildings/barracks.yaml"))
# yaml_files = list(uh_original_objects_path.glob("buildings/bakery.yaml"))
# yaml_files = list(uh_original_objects_path.glob("buildings/mainsquare.yaml"))
# yaml_files = list(uh_original_objects_path.glob("buildings/*.yaml"))
# yaml_files = list(uh_original_objects_path.glob("units/**/*collector.yaml"))
# yaml_files = list(uh_original_objects_path.glob("units/**/settlercollector.yaml"))
# yaml_files = list(uh_original_objects_path.glob("units/**/storagecollector.yaml"))
# yaml_files = list(uh_original_objects_path.glob("units/**/lumberjackcollector.yaml"))
yaml_files = list(uh_original_objects_path.glob("units/**/*.yaml"))
gfx_path = (uh_original_objects_path.parent.parent / "gfx").resolve()
# yaml_files = Path(r"S:\src\unknown-horizons\content\objects\buildings").glob("bakery.yaml")

buildings = [] # list of building infos
unique_uids = set()
for yaml_file in yaml_files:
  print(f"processing {yaml_file}")
  with open(yaml_file, 'r') as fh:
    obj = yaml.safe_load(fh)
  # fix up some fields:
  for production_line in obj["components"].get("ProducerComponent",{}).get("productionlines",{}).values():
    production_line["produces"] = {k: v for k, v in production_line.get("produces",[])} # convert to dict
    production_line["consumes"] = {k: v for k, v in production_line.get("consumes",[])}
  object_name = obj["id"].removeprefix("BUILDINGS.").removeprefix("UNITS.").lower()
  building_or_unit = obj["id"].split(".")[0].capitalize() # eg. 'Unit' or 'Building'
  building_path = objects_save_path / (building_or_unit+"2") / object_name
  building_path.mkdir(exist_ok=True)
  sprite_frames_animations = process_action_sets(obj["actionsets"], object_name, building_path)
  # display(obj)
  unique_source_atlas_paths = set(v["source_atlas_path"] for k, v in sprite_frames_animations.items())
  sprite_frames_tres_path = building_path / f"{object_name}.tres"
  obj["sprite_frames_tres_path"] = PurePosixPath(sprite_frames_tres_path.relative_to(project_path))
  obj["tscn_uid"] = generate_uid(obj["id"][0] + object_name, "tscn")
  obj["tres_uid"] = generate_uid(obj["id"][0] + object_name, "tres")
  if obj["tscn_uid"] in unique_uids or obj["tres_uid"] in unique_uids or obj["tscn_uid"] == obj["tres_uid"]:
    raise Exception(f"DUPLICATE UID: {obj['tscn_uid']}, {obj["tres_uid"]}")
  unique_uids.add(obj["tscn_uid"])
  unique_uids.add(obj["tres_uid"])
  obj["tscn_path"] = building_path / f"{object_name}.tscn"
  obj["tscn_res_path"] = PurePosixPath(obj["tscn_path"].relative_to(project_path))
  obj["get_uuid"] = generate_uid
  write_template(obj["tscn_path"], bakery_tscn_template, obj)
# #   print(rt(bakery_tscn_template, obj))
#   # print(rt(animation_frames_tres_template, {"sprite_frames_animations": sprite_frames_animations, "uid": obj["tres_uid"]}))
  fp2uid = {sfa["source_atlas_path"]: sfa["source_atlas_path_uid"] for sfa in sprite_frames_animations.values() if sfa["source_atlas_path_uid"] != None}
  write_template(sprite_frames_tres_path, animation_frames_tres_template, {"sprite_frames_animations": sprite_frames_animations, "fp2uid": fp2uid, "uid": obj["tres_uid"]})
  buildings.append(obj)

In [ ]:
buildings_config_gd_template = """
extends Object
## The building config has all the information about the buildings required

class_name BuildingConfig

## All the buildings represented in enum state
const building_to_tileset_id = {
  NONE                 =   0,
{% for building in buildings %}
  {{ building["str_id"].ljust(20, " ") }} = {{ building["enum_id"] }},
{% endfor %}
}

## Building enum value to the cost(resource to amount)
static var building_to_cost: Dictionary[Buildings, Dictionary] = {
  Buildings.NONE                : {},
{% for building in buildings %}
  Buildings.{{ building["str_id"].ljust(20, " ") }}: {{ building["buildingcosts"] | tojson | replace('"RES.', 'ResourceConfig.Resources.') | replace('":', ':') }},
{% endfor %}
}
"""
for i, building in enumerate(buildings):
  building["enum_id"] = i+100
  building["str_id"] = building["id"].removeprefix("BUILDINGS.").upper()

built_tileset_tres_template = """
{% for building in buildings %}
[ext_resource type="PackedScene" uid="uid://{{ building["tscn_uid"] }}" path="res://{{ building["tscn_res_path"] }}" id="2_{{ building["str_id"] }}"]
{% endfor %}

[sub_resource type="TileSetScenesCollectionSource" id="TileSetScenesCollectionSource_xv0cf"]
resource_name = "Buildings"
{% for building in buildings %}
scenes/{{ building["enum_id"] }}/scene = ExtResource("2_{{ building["str_id"] }}")
{% endfor %}
"""

print(rt(buildings_config_gd_template, {"buildings": buildings}))
print("To be pasted in BuiltTileSet.tres:")
print(rt(built_tileset_tres_template, {"buildings": buildings}))


In [ ]:
# icons merge
from pathlib import Path
import pandas as pd
import numpy as np
from collections import defaultdict

tabwidgets_path = Path(r"s:\src\unknown-horizons\content\gui\icons\tabwidget")
paths = list(tabwidgets_path.glob("**/*.png"))
df = pd.DataFrame([
    {
        "basename": p.parent.relative_to(tabwidgets_path) / p.stem.rsplit('_', 1)[0],
        "suffix": p.stem.rsplit('_', 1)[1] if len(p.stem.rsplit('_', 1)) > 1 else "",
        "filepath_short": p.relative_to(tabwidgets_path),
        "filepath": p,
        "filepath_size": f"{p.relative_to(tabwidgets_path)}{Image.open(p).size}",
        "size": Image.open(p).size,
    }
    for p in paths
    if p.parent.name != "buildmenu" and p.parent.parent.name != "buildmenu" and p.parent.name != "lumberjackcamp" and not p.stem.startswith("dummy_40")
    # if p.parent.name != "buildmenu" and p.parent.name != "lumberjackcamp" and not p.stem.startswith("dummy_40")
    # if p.parent.name != "lumberjackcamp" and not p.stem.startswith("dummy_40")
])
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
# display(df)
# display(df.groupby("size", as_index=False)["filepath"].agg(lambda x: ", ".join([str(p) for p in x]))) # groupby size
# pivot_df = df.pivot(index=["basename", "size"], columns="suffix", values="size")
pivot_df = df.pivot(index=["basename"], columns=["suffix"], values="filepath")
# pivot_df = df.pivot(index=["basename"], columns=["size"], values="filepath")
# pivot_df = df.pivot_table(index=["basename"], columns=["suffix"], values="filepath", aggfunc=lambda x: ", ".join([str(p) for p in x]))
# pivot_df = df.pivot_table(index=["size"], columns=["suffix"], values="filepath", aggfunc=lambda x: ", ".join([str(p) for p in x]))
# display(pivot_df)
size = np.array([40, 46]) + [2, 2] # separation

total_width = size[0] * pivot_df.shape[1]
total_height = size[1] * pivot_df.shape[0]
frames = []
per_basename_frames = defaultdict(dict)
sprite_img = Image.new("RGBA", (total_width, total_height)) # each animation frame set on a separate row
y = 0
for basename, row in sorted(pivot_df.iterrows()):
  x = 0
  for suffix, filename in row.items():
    frame = {}
    frame["img"] = Image.open(filename)
    img_pos = (x, y) + (size - frame["img"].size) / 2 # center the image
    sprite_img.paste(frame["img"], (int(img_pos[0]), int(img_pos[1])))
    frame["xywh"] = (x+1, y+1, *(size-[2,2]))
    frame["src"] = filename
    frame["id"] = str(filename.relative_to(tabwidgets_path).with_suffix("")).replace("\\","_")
    frames.append(frame)
    # basename_with_folder = str(filename.parent.relative_to(tabwidgets_path) / basename).replace("\\","_")
    per_basename_frames[basename][suffix] = frame
    x += size[0]
  y += size[1]
sprite_img.save(r"s:\src\unknown-horizons-godot-port\Assets\UI\Icons\TabWidget\tabwidget_icons.png")
frames_tscn_template = """
{% for frame in frames %}
[sub_resource type="AtlasTexture" id="{{ frame["id"] }}"]
atlas = ExtResource("42_tabwidget_icons_png")
region = Rect2({{ frame["xywh"][0] }}, {{ frame["xywh"][1] }}, {{ frame["xywh"][2] }}, {{ frame["xywh"][3] }})

{% endfor %}

[node name="AllTabs" instance=ExtResource("1_012xi")]
script = ExtResource("2_wox5d")

{% for basename in per_basename_frames.keys() %}
[node name="{{ basename | replace('\\\\', '_') }}" parent="LeftFloatingPanel/TabSwitches" index="{{ loop.index }}" instance=ExtResource("2_ig0oa")]
texture_normal = SubResource("{{ per_basename_frames[basename]['u']['id'] }}")
texture_pressed = SubResource("{{ per_basename_frames[basename]['d']['id'] }}")
texture_hover = SubResource("{{ per_basename_frames[basename]['h']['id'] }}")
texture_active = SubResource("{{ per_basename_frames[basename]['a']['id'] }}")

{% endfor %}
"""
print(rt(frames_tscn_template, {"frames": frames, "per_basename_frames": per_basename_frames}))